# Fast on-disk perturb-seq preprocessing

Runs the on-disk min10cells perturb-seq preprocessing over every top-level `.h5ad` in `RAW_DIR`, excluding deprecated Southard files. The matrix reader supports both CSR-backed `X` and dense h5ad `X` datasets, so this single notebook covers the hct116/hek293t CSR inputs and the Nadig/Replogle dense inputs.


In [ ]:
import gc
import os
import re
import sys
import traceback
from concurrent.futures import ProcessPoolExecutor, as_completed
from dataclasses import dataclass
from pathlib import Path

import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy import sparse
from sklearn.metrics.pairwise import cosine_similarity
from tqdm.auto import tqdm

PAGE_TORCH_PYTHON = '/opt/conda/envs/page_torch/bin/python'
print('python:', sys.executable)
if Path(sys.executable).resolve() != Path(PAGE_TORCH_PYTHON).resolve():
    print(f'[warn] Expected page_torch python: {PAGE_TORCH_PYTHON}')

RAW_DIR = Path('/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq')
OUT_DIR = Path('/nas/project1/PAGE_PJH/02.processed_perturbseq')
QC_PLOTS_DIR = OUT_DIR / 'qc_plots'
TMP_DIR = Path(os.environ.get('PAGE_PERTURBSEQ_TMP', '/tmp/perturbseq_fast_on_disk_min10cells'))
GENE_METADATA_PATH = Path('/nas/project1/PAGE_PJH/01.rawdata/01.perturb-seq/xatlas_orion/metadata/gene_metadata.parquet')

MIN_CELLS_PER_GENE = 10
MIN_MEAN_UMI = 0.01
CHUNK_SIZE = 2000
OVERWRITE = False

# Set to None to process every non-excluded h5ad. Use a list for a targeted rerun.
DATASET_NAMES = None # ['hct116.h5ad']

WRITE_NTC = True
WRITE_EFFECTS = True
# hct116 all-guide dense float32 temp matrix is about 215 GB. Enable only if downstream needs it.
WRITE_ALL_GUIDES = True
WRITE_QC = True

# Direct CSR reader controls. Larger span means fewer HDF5 calls but more over-read for sparse row selections.
CSR_MAX_SPAN_ROWS = 4096
CSR_MAX_OVERREAD_RATIO = 64.0
GC_EVERY_N_CHUNKS = 25

# Parallelize across datasets only after confirming single-dataset speed and available RAM/NVMe/NAS bandwidth.
DATASET_WORKERS = 1

QC_MAX_SHARED_GENES = 3000
QC_MAX_NTC_CELLS = 10000
QC_MAX_SCATTER_POINTS = 500000
QC_RANDOM_SEED = 0

EXCLUDE_FILENAMES = {
    'southard_rpe1_tf_crispra_effect.h5ad',
    'deprecated_southard_rpe1_tf_crispra.h5ad',
}

DATASET_CONFIGS = {
    'hct116.h5ad': {
        'gene_col': 'gene_target',
        'batch_col': 'sample',
        'control_label': 'Non-Targeting',
    },
    'hek293t.h5ad': {
        'gene_col': 'gene_target',
        'batch_col': 'sample',
        'control_label': 'Non-Targeting',
    },
    'nadig_hepg2.h5ad': {
        'gene_col': 'gene',
        'batch_col': 'gem_group',
        'control_label': 'non-targeting',
    },
    'nadig_jurkat.h5ad': {
        'gene_col': 'gene',
        'batch_col': 'gem_group',
        'control_label': 'non-targeting',
    },
    'replogle_k562_essential.h5ad': {
        'gene_col': 'gene',
        'batch_col': 'batch',
        'control_label': 'non-targeting',
    },
    'replogle_k562_gwps.h5ad': {
        'gene_col': 'gene',
        'batch_col': 'batch',
        'control_label': 'non-targeting',
    },
    'replogle_rpe1.h5ad': {
        'gene_col': 'gene',
        'batch_col': 'batch',
        'control_label': 'non-targeting',
    },
}

OUT_DIR.mkdir(parents=True, exist_ok=True)
QC_PLOTS_DIR.mkdir(parents=True, exist_ok=True)
TMP_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
def iter_chunks(index, chunk_size):
    for start in range(0, len(index), chunk_size):
        yield index[start:start + chunk_size]


def total_chunks(length, chunk_size):
    return (length + chunk_size - 1) // chunk_size


def maybe_gc(chunk_idx):
    if GC_EVERY_N_CHUNKS and (chunk_idx + 1) % GC_EVERY_N_CHUNKS == 0:
        gc.collect()


def make_unique(labels, fallback_index):
    labels = pd.Series(labels, index=pd.Index(fallback_index), dtype='object')
    labels = labels.fillna(labels.index.to_series()).astype(str)
    duplicated = labels.duplicated(keep=False)
    labels.loc[duplicated] = labels.loc[duplicated] + '|' + labels.index[duplicated].astype(str)
    return labels.to_numpy()


def safe_stem(path):
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', Path(path).stem)


def output_paths(input_path):
    stem = safe_stem(input_path)
    return {
        'ntc': OUT_DIR / f'{stem}_min10cells_ntc.h5ad',
        'all_guides': OUT_DIR / f'{stem}_min10cells_allguides_ntc_norm.h5ad',
        'noncentered': OUT_DIR / f'{stem}_min10cells_noncentered_effectsize.csv.gz',
        'centered': OUT_DIR / f'{stem}_min10cells_effectsize.csv.gz',
    }


def expected_output_paths(paths):
    expected = []
    if WRITE_NTC:
        expected.append(paths['ntc'])
    if WRITE_EFFECTS:
        expected.extend([paths['noncentered'], paths['centered']])
    if WRITE_ALL_GUIDES:
        expected.append(paths['all_guides'])
    return expected


def qc_plot_paths(input_path):
    stem = safe_stem(input_path)
    return {
        'co_exp_vs_co_func': QC_PLOTS_DIR / f'{stem}_co_exp_vs_co_func.png',
        'co_reg_vs_co_func': QC_PLOTS_DIR / f'{stem}_co_reg_vs_co_func.png',
    }


def get_config(input_path, obs_columns):
    name = Path(input_path).name
    if name in DATASET_CONFIGS:
        return DATASET_CONFIGS[name].copy()

    gene_candidates = ['gene', 'gene_target', 'target_gene', 'guide_target', 'perturbation']
    batch_candidates = ['batch', 'sample', 'gem_group', 'gemgroup', 'dataset']
    gene_col = next((col for col in gene_candidates if col in obs_columns), None)
    batch_col = next((col for col in batch_candidates if col in obs_columns), None)
    if gene_col is None or batch_col is None:
        raise ValueError(f'Could not infer gene/batch columns for {name}')
    return {'gene_col': gene_col, 'batch_col': batch_col, 'control_label': None}


def infer_control_label(gene_values):
    preferred = ['non-targeting', 'Non-Targeting', 'non', 'control']
    values = pd.Index(pd.Series(gene_values).dropna().astype(str).unique())
    for label in preferred:
        if label in values:
            return label
    matches = [v for v in values if v.lower() in {'non-targeting', 'non', 'control', 'ntc'}]
    if matches:
        return matches[0]
    raise ValueError('Could not infer an exact non-targeting/control label')


def standardize_obs(adata, config):
    obs = adata.obs.copy()
    gene_col = config['gene_col']
    batch_col = config['batch_col']
    if gene_col not in obs.columns:
        raise KeyError(f'Missing gene column: {gene_col}')
    if batch_col not in obs.columns:
        raise KeyError(f'Missing batch column: {batch_col}')

    obs['gene'] = obs[gene_col].astype('object').astype(str)
    obs['batch'] = obs[batch_col].astype('object').astype(str)
    control_label = config.get('control_label') or infer_control_label(obs['gene'])
    return obs, control_label


def load_gene_metadata():
    if not GENE_METADATA_PATH.exists():
        return None
    try:
        return pd.read_parquet(GENE_METADATA_PATH).set_index('ensembl_id')
    except Exception as exc:
        print(f'Could not read gene metadata from {GENE_METADATA_PATH}: {exc}')
        return None


GENE_METADATA = load_gene_metadata()


def prepare_selected_var(adata, selected_gene_idx, mean_umi_per_gene):
    selected_var = adata.var.iloc[selected_gene_idx].copy()
    selected_original_names = pd.Index(adata.var_names[selected_gene_idx])

    selected_var['original_var_name'] = selected_original_names.to_numpy()
    if 'gene_id' not in selected_var.columns:
        selected_var['gene_id'] = selected_original_names.to_numpy()
    if 'ensembl_id' not in selected_var.columns:
        if 'gene_id' in selected_var.columns:
            selected_var['ensembl_id'] = selected_var['gene_id'].astype('object').astype(str).to_numpy()
        else:
            selected_var['ensembl_id'] = selected_original_names.to_numpy()

    if 'gene_name' in selected_var.columns:
        gene_names = pd.Series(selected_var['gene_name'].to_numpy(), index=selected_original_names, dtype='object')
    elif GENE_METADATA is not None and 'gene_name' in GENE_METADATA.columns:
        gene_names = GENE_METADATA['gene_name'].reindex(selected_original_names)
    else:
        gene_names = pd.Series(selected_original_names.to_numpy(), index=selected_original_names, dtype='object')

    selected_var['gene_name'] = gene_names.fillna(pd.Series(selected_original_names.to_numpy(), index=selected_original_names)).astype(str).to_numpy()
    selected_var['gene_name'] = make_unique(selected_var['gene_name'], selected_original_names)
    selected_var['mean_umi_per_filtered_cell'] = mean_umi_per_gene[selected_gene_idx]
    selected_var['keep_min_umi'] = True
    selected_var = selected_var.set_index('gene_name', drop=False)
    return selected_var


def target_medians_for_batches(batches, median_reads_cache, default_median):
    missing = sorted(set(batches) - set(median_reads_cache))
    if missing:
        print(f'using global target median for {len(missing)} batches without perturbation cells: {missing[:5]}')
    return np.asarray([median_reads_cache.get(batch, default_median) for batch in batches], dtype=np.float32)


def remove_temp_file(path):
    try:
        Path(path).unlink(missing_ok=True)
    except TypeError:
        path = Path(path)
        if path.exists():
            path.unlink()


In [ ]:
@dataclass
class CSRReadConfig:
    max_span_rows: int = CSR_MAX_SPAN_ROWS
    max_overread_ratio: float = CSR_MAX_OVERREAD_RATIO


class H5CSRReader:
    """Direct reader for AnnData h5ad files whose X is CSR or dense.

    CSR-backed h5ad files are fastest when row spans are read directly from
    X/data, X/indices, and X/indptr. The failed rerun datasets store X as a
    dense h5py dataset, so this reader also reads dense row spans and converts
    each chunk to CSR to preserve the downstream sparse-matrix code path.
    """

    def __init__(self, path, config=None):
        self.path = Path(path)
        self.config = config or CSRReadConfig()
        self.file = h5py.File(self.path, 'r')
        if 'X' not in self.file:
            raise KeyError(f'{self.path} has no X group/dataset')

        x = self.file['X']
        encoding = x.attrs.get('encoding-type')
        if isinstance(encoding, bytes):
            encoding = encoding.decode()
        self.encoding = encoding

        if isinstance(x, h5py.Group) and encoding == 'csr_matrix':
            self.storage = 'csr_matrix'
            self.x = x
            self.data = x['data']
            self.indices = x['indices']
            self.indptr = x['indptr']
            self.shape = tuple(int(v) for v in x.attrs['shape'])
        elif isinstance(x, h5py.Dataset):
            self.storage = 'dense_array'
            self.x = x
            self.shape = tuple(int(v) for v in x.shape)
        else:
            raise NotImplementedError(
                f'{self.path} X encoding is {encoding!r} and storage is {type(x).__name__}; '
                'expected csr_matrix group or dense dataset'
            )

        if len(self.shape) != 2:
            raise ValueError(f'{self.path} X must be two-dimensional, got shape {self.shape}')
        self.n_obs, self.n_vars = self.shape

    def close(self):
        self.file.close()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()

    def _normalize_positions(self, positions):
        positions = np.asarray(positions, dtype=np.int64)
        if positions.ndim != 1:
            raise ValueError('positions must be one-dimensional')
        if len(positions) == 0:
            return positions
        if positions[0] < 0 or positions[-1] >= self.n_obs:
            raise IndexError('row positions are out of bounds')
        if np.any(np.diff(positions) < 0):
            raise ValueError('positions must be sorted ascending')
        return positions

    def _normalize_var_indices(self, var_indices):
        if var_indices is None:
            return None, self.n_vars
        var_indices = np.asarray(var_indices, dtype=np.int64)
        if var_indices.ndim != 1:
            raise ValueError('var_indices must be one-dimensional')
        if len(var_indices) == 0:
            return var_indices, 0
        if var_indices.min() < 0 or var_indices.max() >= self.n_vars:
            raise IndexError('var_indices are out of bounds')
        return var_indices, len(var_indices)

    def _iter_segments(self, positions):
        n = len(positions)
        start = 0
        while start < n:
            end = start + 1
            while end < n:
                span = int(positions[end] - positions[start] + 1)
                count = end - start + 1
                if span > self.config.max_span_rows:
                    break
                if count >= 16 and span / count > self.config.max_overread_ratio:
                    break
                end += 1
            yield positions[start:end]
            start = end

    def _make_col_lookup(self, var_indices, n_cols):
        if var_indices is None:
            return None
        lookup = np.full(self.n_vars, -1, dtype=np.int32)
        if len(var_indices):
            lookup[var_indices] = np.arange(n_cols, dtype=np.int32)
        return lookup

    def _read_csr_segment(self, rows, col_lookup, n_cols):
        n_rows = len(rows)
        if n_rows == 0:
            return sparse.csr_matrix((0, n_cols), dtype=np.float32)

        lo = int(rows[0])
        hi = int(rows[-1])
        ptr = np.asarray(self.indptr[lo:hi + 2], dtype=np.int64)
        base = int(ptr[0])
        stop = int(ptr[-1])
        local_ptr = ptr - base
        lengths = np.diff(local_ptr)
        span_len = hi - lo + 1

        if stop == base or n_cols == 0:
            return sparse.csr_matrix((n_rows, n_cols), dtype=np.float32)

        data = np.asarray(self.data[base:stop], dtype=np.float32)
        indices = np.asarray(self.indices[base:stop], dtype=np.int32)
        row_ids = np.repeat(np.arange(span_len, dtype=np.int32), lengths)

        if col_lookup is None:
            cols = indices
        else:
            mapped = col_lookup[indices]
            keep = mapped >= 0
            if not np.any(keep):
                return sparse.csr_matrix((n_rows, n_cols), dtype=np.float32)
            data = data[keep]
            cols = mapped[keep].astype(np.int32, copy=False)
            row_ids = row_ids[keep]

        if n_rows != span_len:
            row_lookup = np.full(span_len, -1, dtype=np.int32)
            row_lookup[rows - lo] = np.arange(n_rows, dtype=np.int32)
            local_rows = row_lookup[row_ids]
            keep_rows = local_rows >= 0
            if not np.any(keep_rows):
                return sparse.csr_matrix((n_rows, n_cols), dtype=np.float32)
            data = data[keep_rows]
            cols = cols[keep_rows]
            local_rows = local_rows[keep_rows]
        else:
            local_rows = row_ids

        counts = np.bincount(local_rows, minlength=n_rows)
        out_indptr = np.empty(n_rows + 1, dtype=np.int64)
        out_indptr[0] = 0
        np.cumsum(counts, out=out_indptr[1:])
        return sparse.csr_matrix((data, cols, out_indptr), shape=(n_rows, n_cols), dtype=np.float32)

    def _read_dense_segment(self, rows, var_indices, n_cols):
        n_rows = len(rows)
        if n_rows == 0 or n_cols == 0:
            return sparse.csr_matrix((n_rows, n_cols), dtype=np.float32)

        lo = int(rows[0])
        hi = int(rows[-1])
        dense = np.asarray(self.x[lo:hi + 1, :], dtype=np.float32)
        if n_rows != dense.shape[0]:
            dense = dense[rows - lo, :]
        if var_indices is not None:
            dense = dense[:, var_indices]
        return sparse.csr_matrix(dense, dtype=np.float32)

    def read_positions(self, positions, var_indices=None):
        positions = self._normalize_positions(positions)
        var_indices, n_cols = self._normalize_var_indices(var_indices)
        if len(positions) == 0:
            return sparse.csr_matrix((0, n_cols), dtype=np.float32)

        if self.storage == 'csr_matrix':
            col_lookup = self._make_col_lookup(var_indices, n_cols)
            chunks = [self._read_csr_segment(seg, col_lookup, n_cols) for seg in self._iter_segments(positions)]
        else:
            chunks = [self._read_dense_segment(seg, var_indices, n_cols) for seg in self._iter_segments(positions)]

        if len(chunks) == 1:
            return chunks[0]
        return sparse.vstack(chunks, format='csr')


In [ ]:
def choose_qc_genes(feature_names, guide_names):
    feature_index = pd.Index(feature_names)
    guide_index = pd.Index(guide_names)
    shared = feature_index[feature_index.isin(guide_index)]
    if len(shared) > QC_MAX_SHARED_GENES:
        rng = np.random.default_rng(QC_RANDOM_SEED)
        keep = np.sort(rng.choice(len(shared), size=QC_MAX_SHARED_GENES, replace=False))
        shared = shared[keep]
    return pd.Index(shared)


def choose_qc_rows(n_rows):
    if n_rows <= QC_MAX_NTC_CELLS:
        return np.arange(n_rows, dtype=np.int64)
    rng = np.random.default_rng(QC_RANDOM_SEED)
    return np.sort(rng.choice(n_rows, size=QC_MAX_NTC_CELLS, replace=False)).astype(np.int64)


def subset_matrix_rows_cols(matrix, row_pos, col_pos, row_chunk_size=1000):
    chunks = []
    for rows in iter_chunks(row_pos, row_chunk_size):
        x = matrix[rows, :]
        if sparse.issparse(x):
            x = x[:, col_pos].toarray()
        else:
            x = np.asarray(x)[:, col_pos]
        chunks.append(x.astype(np.float32, copy=False))
    if not chunks:
        return np.empty((0, len(col_pos)), dtype=np.float32)
    return np.vstack(chunks)


def save_similarity_scatter(x_matrix, y_matrix, x_label, y_label, title, output_path):
    tri = np.triu_indices(x_matrix.shape[0], k=1)
    x = np.asarray(x_matrix[tri], dtype=np.float32)
    y = np.asarray(y_matrix[tri], dtype=np.float32)
    keep = np.isfinite(x) & np.isfinite(y) & (x != 1) & (y != 1)
    x = x[keep]
    y = y[keep]

    if len(x) > QC_MAX_SCATTER_POINTS:
        rng = np.random.default_rng(QC_RANDOM_SEED)
        sample_idx = rng.choice(len(x), size=QC_MAX_SCATTER_POINTS, replace=False)
        x = x[sample_idx]
        y = y[sample_idx]

    corr = np.corrcoef(x, y)[0, 1] if len(x) > 1 else np.nan
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.scatter(x, y, alpha=0.18, s=1, rasterized=True)
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.set_title(f'{title}\nr={corr:.3f}, n={len(x):,}')
    ax.grid(False)
    fig.tight_layout()
    fig.savefig(output_path, dpi=200, bbox_inches='tight')
    plt.close(fig)


def save_qc_plots(input_path, ko_means, ntc_values, feature_names, qc_paths):
    try:
        shared_genes = choose_qc_genes(feature_names, ko_means.index)
        if len(shared_genes) < 3:
            print(f'[qc] {Path(input_path).name}: skipped; only {len(shared_genes)} shared genes')
            return

        feature_index = pd.Index(feature_names)
        feature_pos = feature_index.get_indexer(shared_genes)
        row_pos = choose_qc_rows(ntc_values.shape[0])
        ntc_subset = subset_matrix_rows_cols(ntc_values, row_pos, feature_pos)

        co_expression = cosine_similarity(ntc_subset.T)
        co_function = cosine_similarity(ko_means.loc[shared_genes, :].to_numpy(dtype=np.float32, copy=False))
        co_regulation = cosine_similarity(ko_means.loc[:, shared_genes].T.to_numpy(dtype=np.float32, copy=False))

        stem = safe_stem(input_path)
        suffix = f'{len(shared_genes):,} genes, {len(row_pos):,} NTC cells'
        save_similarity_scatter(
            co_expression,
            co_function,
            f'{stem} co_expression',
            f'{stem} co_function',
            f'{stem}: co_expression vs co_function ({suffix})',
            qc_paths['co_exp_vs_co_func'],
        )
        save_similarity_scatter(
            co_regulation,
            co_function,
            f'{stem} co_regulation',
            f'{stem} co_function',
            f'{stem}: co_regulation vs co_function ({suffix})',
            qc_paths['co_reg_vs_co_func'],
        )
        print('wrote:', qc_paths['co_exp_vs_co_func'])
        print('wrote:', qc_paths['co_reg_vs_co_func'])
        del ntc_subset, co_expression, co_function, co_regulation
        gc.collect()
    except Exception as exc:
        print(f'[qc] {Path(input_path).name}: plot generation failed: {exc}')


def save_qc_plots_from_outputs(input_path, paths, qc_paths):
    if all(path.exists() for path in qc_paths.values()):
        return
    if not paths['centered'].exists() or not paths['ntc'].exists():
        return
    try:
        ko_means = pd.read_csv(paths['centered'], index_col=0)
        adata_ntc = sc.read_h5ad(paths['ntc'], backed='r')
        try:
            save_qc_plots(input_path, ko_means, adata_ntc.X, adata_ntc.var_names.to_numpy(), qc_paths)
        finally:
            try:
                adata_ntc.file.close()
            except Exception:
                pass
        del ko_means
        gc.collect()
    except Exception as exc:
        print(f'[qc] {Path(input_path).name}: could not load existing outputs for QC: {exc}')


In [ ]:
def process_h5ad(input_path, overwrite=OVERWRITE):
    input_path = Path(input_path)
    paths = output_paths(input_path)
    qc_paths = qc_plot_paths(input_path)
    expected_paths = expected_output_paths(paths)

    if not overwrite and expected_paths and all(path.exists() for path in expected_paths):
        if WRITE_QC and WRITE_NTC and WRITE_EFFECTS and not all(path.exists() for path in qc_paths.values()):
            print(f'[qc] {input_path.name}: data outputs already exist; generating missing QC plots')
            save_qc_plots_from_outputs(input_path, paths, qc_paths)
            return {'input': input_path.name, 'status': 'qc_done'}
        print(f'[skip] {input_path.name}: expected outputs already exist')
        return {'input': input_path.name, 'status': 'skipped'}

    print(f'\n=== {input_path.name} ===')
    print('outputs:', {key: str(value) for key, value in paths.items()})
    print('write_all_guides:', WRITE_ALL_GUIDES, 'tmp_dir:', TMP_DIR)

    adata = sc.read_h5ad(input_path, backed='r')
    temp_paths = []
    try:
        with H5CSRReader(input_path) as reader:
            if reader.shape != adata.shape:
                raise ValueError(f'X shape mismatch: h5={reader.shape}, adata={adata.shape}')

            config = get_config(input_path, adata.obs.columns)
            obs, control_label = standardize_obs(adata, config)
            print('shape:', adata.shape)
            print('X storage:', reader.storage, 'encoding:', reader.encoding)
            print('gene_col:', config['gene_col'], 'batch_col:', config['batch_col'], 'control_label:', control_label)

            valid_gene_mask = obs['gene'].notna() & obs['gene'].ne('nan') & obs['gene'].ne('None') & obs['gene'].ne('')
            gene_counts = (
                obs.loc[valid_gene_mask, ['gene']]
                .groupby('gene', as_index=False)
                .size()
                .rename(columns={'size': 'n_cells'})
                .sort_values('n_cells', ascending=False)
            )
            blacklist = gene_counts.loc[gene_counts['n_cells'] < MIN_CELLS_PER_GENE, 'gene'].tolist()
            filt_mask = (valid_gene_mask & ~obs['gene'].isin(blacklist)).to_numpy()
            filt_positions = np.flatnonzero(filt_mask)

            control_mask = filt_mask & obs['gene'].eq(control_label).to_numpy()
            pert_mask = filt_mask & obs['gene'].ne(control_label).to_numpy()
            control_positions = np.flatnonzero(control_mask)
            pert_positions = np.flatnonzero(pert_mask)

            print(f'#blacklist genes (<{MIN_CELLS_PER_GENE} cells):', len(blacklist))
            print(f'filtered cells after guide blacklist: {len(filt_positions)}/{adata.n_obs}')
            print('perturbation cells:', len(pert_positions), 'control cells:', len(control_positions))
            if len(control_positions) == 0:
                raise ValueError(f'No control cells found for label {control_label!r}')
            if len(pert_positions) == 0:
                raise ValueError('No perturbation cells remain after filtering')

            gene_sum = np.zeros(adata.n_vars, dtype=np.float64)
            for chunk_idx, chunk_positions in enumerate(tqdm(
                iter_chunks(filt_positions, CHUNK_SIZE),
                total=total_chunks(len(filt_positions), CHUNK_SIZE),
                desc=f'{input_path.stem}: mean_umi',
            )):
                x = reader.read_positions(chunk_positions)
                gene_sum += np.asarray(x.sum(axis=0)).ravel().astype(np.float64, copy=False)
                del x
                maybe_gc(chunk_idx)

            mean_umi_per_gene = gene_sum / max(len(filt_positions), 1)
            gene_mask = mean_umi_per_gene > MIN_MEAN_UMI
            selected_gene_idx = np.flatnonzero(gene_mask)
            selected_var = prepare_selected_var(adata, selected_gene_idx, mean_umi_per_gene)
            selected_feature_names = selected_var.index.to_numpy()
            n_features = len(selected_gene_idx)
            print(f'genes with mean UMI > {MIN_MEAN_UMI} per filtered cell: {n_features}/{adata.n_vars}')

            matrix_obs = obs.loc[pert_mask].copy()
            control_matrix_obs = obs.loc[control_mask].copy()
            pert_batches = matrix_obs['batch'].to_numpy()
            control_batches = control_matrix_obs['batch'].to_numpy()
            pert_guides = matrix_obs['gene'].to_numpy()

            def read_selected_chunk(obs_indexer):
                return reader.read_positions(obs_indexer, selected_gene_idx)

            batch_reads_per_bc = np.empty(len(pert_positions), dtype=np.float32)
            total_pert_chunks = total_chunks(len(pert_positions), CHUNK_SIZE)
            for chunk_idx, start in enumerate(tqdm(
                range(0, len(pert_positions), CHUNK_SIZE),
                total=total_pert_chunks,
                desc=f'{input_path.stem}: batch_median',
            )):
                end = min(start + CHUNK_SIZE, len(pert_positions))
                x = read_selected_chunk(pert_positions[start:end])
                batch_reads_per_bc[start:end] = np.asarray(x.sum(axis=1)).ravel().astype(np.float32, copy=False)
                del x
                maybe_gc(chunk_idx)

            pert_batch_codes, batch_order = pd.factorize(pert_batches, sort=False)
            median_reads_cache = {}
            for batch_code, batch in enumerate(batch_order):
                batch_mask = pert_batch_codes == batch_code
                reads_per_bc = batch_reads_per_bc[batch_mask]
                median_reads_cache[batch] = float(np.median(reads_per_bc)) if len(reads_per_bc) else 0.0
                print('gemgroup', batch, ':', (int(batch_mask.sum()), n_features), 'median_reads', median_reads_cache[batch])
            global_target_median = float(np.median(batch_reads_per_bc)) if len(batch_reads_per_bc) else 0.0
            del batch_reads_per_bc
            gc.collect()

            control_target_medians = target_medians_for_batches(control_batches, median_reads_cache, global_target_median)
            pert_target_medians = target_medians_for_batches(pert_batches, median_reads_cache, global_target_median)

            control_sum = np.zeros(n_features, dtype=np.float64)
            control_sumsq = np.zeros(n_features, dtype=np.float64)
            control_n = 0
            total_control_chunks = total_chunks(len(control_positions), CHUNK_SIZE)
            for chunk_idx, start in enumerate(tqdm(
                range(0, len(control_positions), CHUNK_SIZE),
                total=total_control_chunks,
                desc=f'{input_path.stem}: control_stats',
            )):
                end = min(start + CHUNK_SIZE, len(control_positions))
                x = read_selected_chunk(control_positions[start:end])
                control_reads_per_bc = np.asarray(x.sum(axis=1)).ravel().astype(np.float32, copy=False)
                control_scaling_factors = np.divide(
                    control_target_medians[start:end],
                    control_reads_per_bc,
                    out=np.ones_like(control_reads_per_bc, dtype=np.float32),
                    where=control_reads_per_bc != 0,
                )
                x = x.multiply(control_scaling_factors[:, None]).tocsr()
                control_sum += np.asarray(x.sum(axis=0)).ravel().astype(np.float64, copy=False)
                control_sumsq += np.asarray(x.power(2).sum(axis=0)).ravel().astype(np.float64, copy=False)
                control_n += x.shape[0]
                del x, control_reads_per_bc, control_scaling_factors
                maybe_gc(chunk_idx)

            control_mean = (control_sum / max(control_n, 1)).astype(np.float32)
            control_var = (control_sumsq - control_n * np.square(control_mean.astype(np.float64))) / max(control_n - 1, 1)
            control_std = np.sqrt(np.clip(control_var, a_min=0, a_max=None)).astype(np.float32)
            control_std[~np.isfinite(control_std)] = 1
            control_std[control_std == 0] = 1

            control_matrix = None
            ntc_memmap_path = TMP_DIR / f'{safe_stem(input_path)}.ntc_zscore.float32.dat'
            if WRITE_NTC or WRITE_QC:
                temp_paths.append(ntc_memmap_path)
                remove_temp_file(ntc_memmap_path)
                ntc_size_gb = control_matrix_obs.shape[0] * n_features * 4 / 1e9
                print(f'Building control AnnData matrix on disk (~{ntc_size_gb:.1f} GB raw float32)...')
                control_matrix = np.memmap(
                    ntc_memmap_path,
                    dtype='float32',
                    mode='w+',
                    shape=(control_matrix_obs.shape[0], n_features),
                )
                control_row = 0
                for chunk_idx, start in enumerate(tqdm(
                    range(0, len(control_positions), CHUNK_SIZE),
                    total=total_control_chunks,
                    desc=f'{input_path.stem}: control_zscore',
                )):
                    end = min(start + CHUNK_SIZE, len(control_positions))
                    x = read_selected_chunk(control_positions[start:end])
                    control_reads_per_bc = np.asarray(x.sum(axis=1)).ravel().astype(np.float32, copy=False)
                    control_scaling_factors = np.divide(
                        control_target_medians[start:end],
                        control_reads_per_bc,
                        out=np.ones_like(control_reads_per_bc, dtype=np.float32),
                        where=control_reads_per_bc != 0,
                    )
                    x = x.multiply(control_scaling_factors[:, None]).tocsr()
                    dense_chunk = x.toarray().astype(np.float32, copy=False)
                    dense_chunk -= control_mean[None, :]
                    dense_chunk /= control_std[None, :]
                    next_row = control_row + dense_chunk.shape[0]
                    control_matrix[control_row:next_row] = dense_chunk
                    control_row = next_row
                    del x, control_reads_per_bc, control_scaling_factors, dense_chunk
                    maybe_gc(chunk_idx)
                control_matrix.flush()

                if WRITE_NTC:
                    control_matrix_for_h5ad = np.asarray(control_matrix, dtype=np.float32)
                    adata_ntc = sc.AnnData(control_matrix_for_h5ad, obs=control_matrix_obs.copy(), var=selected_var.copy())
                    adata_ntc.write_h5ad(paths['ntc'])
                    print('wrote:', paths['ntc'], 'shape:', adata_ntc.shape)
                    del adata_ntc, control_matrix_for_h5ad
                    gc.collect()

            if WRITE_EFFECTS:
                print('Accumulating perturbation guide means...')
                guide_codes, guide_order = pd.factorize(pert_guides, sort=True)
                guide_sum = np.zeros((len(guide_order), n_features), dtype=np.float64)
                guide_count = np.bincount(guide_codes, minlength=len(guide_order)).astype(np.int64)
                for chunk_idx, start in enumerate(tqdm(
                    range(0, len(pert_positions), CHUNK_SIZE),
                    total=total_pert_chunks,
                    desc=f'{input_path.stem}: perturb_zscore',
                )):
                    end = min(start + CHUNK_SIZE, len(pert_positions))
                    x = read_selected_chunk(pert_positions[start:end])
                    reads_per_bc = np.asarray(x.sum(axis=1)).ravel().astype(np.float32, copy=False)
                    scaling_factors = np.divide(
                        pert_target_medians[start:end],
                        reads_per_bc,
                        out=np.ones_like(reads_per_bc, dtype=np.float32),
                        where=reads_per_bc != 0,
                    )
                    x = x.multiply(scaling_factors[:, None]).tocsr()

                    chunk_codes = guide_codes[start:end]
                    unique_codes, inverse = np.unique(chunk_codes, return_inverse=True)
                    group_matrix = sparse.csr_matrix(
                        (np.ones(len(chunk_codes), dtype=np.float32), (inverse, np.arange(len(chunk_codes)))),
                        shape=(len(unique_codes), len(chunk_codes)),
                    )
                    guide_sum[unique_codes] += (group_matrix @ x).toarray().astype(np.float64, copy=False)
                    del x, reads_per_bc, scaling_factors, chunk_codes, unique_codes, inverse, group_matrix
                    maybe_gc(chunk_idx)

                ko_means_noncentered_values = guide_sum / guide_count[:, None]
                ko_means_noncentered_values -= control_mean[None, :].astype(np.float64, copy=False)
                ko_means_noncentered_values /= control_std[None, :].astype(np.float64, copy=False)
                ko_means_noncentered = pd.DataFrame(
                    ko_means_noncentered_values.astype(np.float32),
                    index=guide_order,
                    columns=selected_feature_names,
                )
                ko_means_noncentered.to_csv(paths['noncentered'])
                ko_means = ko_means_noncentered - ko_means_noncentered.mean(axis=0)
                ko_means.to_csv(paths['centered'])
                print('wrote:', paths['noncentered'])
                print('wrote:', paths['centered'])

                if WRITE_QC and control_matrix is not None:
                    save_qc_plots(input_path, ko_means, control_matrix, selected_feature_names, qc_paths)

                del guide_sum, guide_count, ko_means_noncentered_values, ko_means_noncentered, ko_means
                gc.collect()

            if control_matrix is not None:
                del control_matrix
                gc.collect()
            remove_temp_file(ntc_memmap_path)

            if WRITE_ALL_GUIDES:
                all_memmap_path = TMP_DIR / f'{safe_stem(input_path)}.allguides_ntc_zscore.float32.dat'
                temp_paths.append(all_memmap_path)
                remove_temp_file(all_memmap_path)
                all_positions = filt_positions
                all_matrix_obs = obs.loc[filt_mask].copy()
                all_batches = all_matrix_obs['batch'].to_numpy()
                all_target_medians = target_medians_for_batches(all_batches, median_reads_cache, global_target_median)

                all_size_gb = all_matrix_obs.shape[0] * n_features * 4 / 1e9
                print(f'Building all-guide AnnData matrix on disk (~{all_size_gb:.1f} GB raw float32)...')
                all_matrix = np.memmap(
                    all_memmap_path,
                    dtype='float32',
                    mode='w+',
                    shape=(all_matrix_obs.shape[0], n_features),
                )
                all_row = 0
                total_all_chunks = total_chunks(len(all_positions), CHUNK_SIZE)
                for chunk_idx, start in enumerate(tqdm(
                    range(0, len(all_positions), CHUNK_SIZE),
                    total=total_all_chunks,
                    desc=f'{input_path.stem}: all_guides_zscore',
                )):
                    end = min(start + CHUNK_SIZE, len(all_positions))
                    x = read_selected_chunk(all_positions[start:end])
                    reads_per_bc = np.asarray(x.sum(axis=1)).ravel().astype(np.float32, copy=False)
                    scaling_factors = np.divide(
                        all_target_medians[start:end],
                        reads_per_bc,
                        out=np.ones_like(reads_per_bc, dtype=np.float32),
                        where=reads_per_bc != 0,
                    )
                    x = x.multiply(scaling_factors[:, None]).tocsr()
                    dense_chunk = x.toarray().astype(np.float32, copy=False)
                    dense_chunk -= control_mean[None, :]
                    dense_chunk /= control_std[None, :]
                    next_row = all_row + dense_chunk.shape[0]
                    all_matrix[all_row:next_row] = dense_chunk
                    all_row = next_row
                    del x, reads_per_bc, scaling_factors, dense_chunk
                    maybe_gc(chunk_idx)
                all_matrix.flush()

                all_matrix_for_h5ad = np.asarray(all_matrix, dtype=np.float32)
                adata_all_guides = sc.AnnData(all_matrix_for_h5ad, obs=all_matrix_obs.copy(), var=selected_var.copy())
                adata_all_guides.write_h5ad(paths['all_guides'])
                print('wrote:', paths['all_guides'], 'shape:', adata_all_guides.shape)
                del adata_all_guides, all_matrix_for_h5ad, all_matrix, all_matrix_obs, all_target_medians
                gc.collect()
                remove_temp_file(all_memmap_path)

            return {
                'input': input_path.name,
                'status': 'done',
                'n_obs': int(adata.n_obs),
                'n_vars': int(adata.n_vars),
                'n_filtered_cells': int(len(filt_positions)),
                'n_control_cells': int(len(control_positions)),
                'n_perturbation_cells': int(len(pert_positions)),
                'n_selected_features': int(n_features),
                'n_guides': int(len(pd.Index(pert_guides).unique())),
                'write_all_guides': bool(WRITE_ALL_GUIDES),
            }
    finally:
        try:
            adata.file.close()
        except Exception:
            pass
        for temp_path in temp_paths:
            remove_temp_file(temp_path)
        gc.collect()


In [ ]:
input_paths = [
    path for path in sorted(RAW_DIR.glob('*.h5ad'))
    if path.name not in EXCLUDE_FILENAMES
]
if DATASET_NAMES is not None:
    wanted = set(DATASET_NAMES)
    input_paths = [path for path in input_paths if path.name in wanted]
    missing = wanted - {path.name for path in input_paths}
    if missing:
        raise FileNotFoundError(f'DATASET_NAMES not found under RAW_DIR: {sorted(missing)}')

print('Input files:')
for path in input_paths:
    print('-', path.name)

print('DATASET_WORKERS:', DATASET_WORKERS)
print('WRITE_ALL_GUIDES:', WRITE_ALL_GUIDES)


In [ ]:
results = []
if DATASET_WORKERS <= 1:
    for input_path in input_paths:
        try:
            result = process_h5ad(input_path)
        except Exception as exc:
            print(f'[failed] {input_path.name}: {exc}')
            traceback.print_exc()
            result = {'input': input_path.name, 'status': 'failed', 'error': repr(exc)}
        results.append(result)
        gc.collect()
else:
    print('[parallel] Running one process per dataset. Keep this small on shared NAS.')
    with ProcessPoolExecutor(max_workers=DATASET_WORKERS) as executor:
        future_to_path = {executor.submit(process_h5ad, path): path for path in input_paths}
        for future in as_completed(future_to_path):
            input_path = future_to_path[future]
            try:
                result = future.result()
            except Exception as exc:
                print(f'[failed] {input_path.name}: {exc}')
                traceback.print_exc()
                result = {'input': input_path.name, 'status': 'failed', 'error': repr(exc)}
            results.append(result)

results_df = pd.DataFrame(results)
results_df
